# Sprint 4 - FieldCare project workspace

This is the shared student notebook for the Sprint 4 FieldCare project. Keep one copy for the whole sprint: explore the data, scope a useful slice, define the response contract, build the application path, inspect traces, stress-test edge cases, and record one targeted improvement.

FieldCare supports field-service technicians who diagnose equipment issues. The project application combines service documentation, retrieval, reranking, structured equipment and ticket data, registered tools, context assembly, response control, abstention, and escalation behavior.

The most important habit in this project is traceability: before trusting an answer, inspect the data, the retrieved evidence, the tool results, the decision flags, and the response object.

Before you start:

- Choose **File > Save a copy in Drive** before editing in Colab.
- Keep one notebook copy for all Sprint 4 Campus and live-session work.
- Keep the source data unchanged; make project changes in the named editable cells.
- A funded OpenRouter key is required. Retrieval uses real embeddings and reranking; each application response is generated by a model from the retrieved passages and actual local tool results. Runs consume API credits.

By the end, your notebook should show a scoped application boundary, a measurable response contract, one complete FieldCare trace, one edge-case trace, and one improvement note with before-and-after evidence.

**Resource type:** Student project notebook used across all Sprint 4 Campus lessons and live sessions.


## 1. Install helper core and notebook dependencies

Run this first in Colab. The helper core is installed from the course GitHub repository so retrieval helpers and tool execution match earlier notebooks. If imports fail after installation, choose **Runtime > Restart session**, then run sections 1 and 2 again.


In [ ]:
#@title Install helper core from GitHub
%pip install -q --force-reinstall --no-cache-dir "ms-ai-ml-helper-core @ git+https://github.com/richhiey/ai-app-dev_Mod-A.git@main"
%pip install -q "pandas>=2,<3"

## 2. Imports, credentials, and the application path

The application code is in `src/fieldcare_project.py`. The visible cells build and inspect each stage: input contract, MCP retrieval, local-data tools, source filtering, a live structured LLM response, and evaluation.

The course dataset describes fictional equipment. Tools really read those records; no external equipment system is connected. Embeddings, hybrid retrieval, reranking, and response generation use real APIs. Missing credentials or failed calls cannot be replaced by canned results.


In [ ]:
#@title Imports and credentials
import copy
import json
import pandas as pd
from IPython.display import display, Markdown
from notebook_setup import require_openrouter_key
from fieldcare_project import (
    accepted_evidence,
    assemble_fieldcare_context_from_parts,
    assemble_pipeline_design,
    build_fieldcare_pipeline,
    default_pipeline_design,
    draft_application_response,
    evaluate_fieldcare_pipeline,
    execute_tool_plan,
    load_fieldcare_environment,
    make_application_scope,
    make_custom_request,
    pretty,
    request_by_id,
    run_eval_case,
    run_fieldcare_app,
    show_context_trace,
    show_environment_overview,
    show_evaluation_results,
    show_extension_points,
    show_fieldcare_data_profile,
    show_orchestration_summary,
    show_processed_project_views,
    show_registered_tools,
    show_rerank_config_guide,
    show_retrieval_rules,
    show_tool_plan,
)

_ = require_openrouter_key(prompt=True)

## 3. Load and inspect the FieldCare data

Start like a data scientist or ML engineer: inspect the data before building logic on top of it. The processing code is hidden in helper functions, but the outputs show the project grain, document coverage, structured records, request families, and relationship checks.

Action: run both cells in this section before making design choices.

Expected output: loaded assets, document coverage, equipment and ticket profiles, request families, relationship checks, a request workbench, a document catalog, and a tool contract catalog.

Check: you can name which answers need `service_docs.jsonl`, which need tool state, and which need both.


In [ ]:
#@title Load the FieldCare data package
fieldcare_env = load_fieldcare_environment()
show_environment_overview(fieldcare_env)

In [ ]:
#@title Explore and process the FieldCare dataset { display-mode: "form" }
fieldcare_profile = show_fieldcare_data_profile(fieldcare_env)
show_processed_project_views(fieldcare_profile)

## 4. Load and validate the three sprint artifacts

Upload the JSON files you exported in Sprints 1–3. Sprint 4 uses them as executable project inputs: the app boundary, retrieval decisions, and capability boundaries must all validate before the final integration path is built.


In [ ]:
from pathlib import Path
from fieldcare import load_fieldcare_artifact

ARTIFACT_TYPES = {
    "fieldcare_app_contract.json": "app_contract",
    "fieldcare_retrieval_config.json": "retrieval_config",
    "fieldcare_capability_contracts.json": "capability_contracts",
}
missing = [name for name in ARTIFACT_TYPES if not Path(name).is_file()]
if missing:
    try:
        from google.colab import files

        print("Upload the three sprint artifacts:", ", ".join(missing))
        files.upload()
    except ImportError as exc:
        raise FileNotFoundError(
            "Missing project artifacts: " + ", ".join(missing)
        ) from exc

APP_CONTRACT = load_fieldcare_artifact("fieldcare_app_contract.json", "app_contract")
RETRIEVAL_CONTRACT = load_fieldcare_artifact(
    "fieldcare_retrieval_config.json", "retrieval_config"
)
CAPABILITY_CONTRACT = load_fieldcare_artifact(
    "fieldcare_capability_contracts.json", "capability_contracts"
)
pretty({"validated_artifacts": list(ARTIFACT_TYPES)})

## 5. Choose a useful first request path

A good project starts with one request path that is narrow enough to prove. For the guided run, use `REQ-FC-001`: it needs current service documents, equipment state, maintenance history, warranty state, ticket state, escalation logic, context assembly, and response control.

Action: keep the default first, then switch the request only after you can explain why this path is a strong integrated slice.

Expected output: one request row from the workbench and the full request object.

Check: the selected request has a clear `request_type`, known identifiers, and expected evidence sources.


In [ ]:
#@title Pick the guided request
GUIDED_REQUEST_ID = "REQ-FC-001"  #@param ["REQ-FC-001", "REQ-FC-002", "REQ-FC-003", "REQ-FC-006", "REQ-FC-007", "REQ-FC-010", "REQ-FC-011", "REQ-FC-012", "REQ-FC-013", "REQ-FC-014"]
guided_request = request_by_id(fieldcare_env, GUIDED_REQUEST_ID)
display(
    fieldcare_profile["request_workbench"].loc[
        fieldcare_profile["request_workbench"]["request_id"] == GUIDED_REQUEST_ID,
        [
            "request_id",
            "request_type",
            "difficulty",
            "known_identifiers",
            "model",
            "warranty_status",
            "current_status",
            "expected_sources",
            "preferred_behavior",
        ],
    ]
)
pretty(guided_request)

## 6. Define the app boundary and response contract

The scope tells the app what it should handle. The response contract tells the app what it must return. Keep these simple enough to inspect from notebook output.

Best practice: add only fields and request types that you can trace through data, retrieved documents, tool results, and decision flags.

Action: read the default scope before editing. Then make one intentional change only if your project goal requires it.

Expected output: a JSON object with supported request types, out-of-scope request types, response fields, and success criteria.

Check: every supported request type has a believable evidence path. If you cannot name the needed documents or tools, leave it out of scope for now.


In [ ]:
#@title Apply the validated Sprint 1 contract
SUPPORTED_REQUEST_TYPES = APP_CONTRACT["supported_request_types"]
OUT_OF_SCOPE_REQUEST_TYPES = APP_CONTRACT.get("out_of_scope_request_types", [])
APPLICATION_SCOPE = make_application_scope(
    SUPPORTED_REQUEST_TYPES, OUT_OF_SCOPE_REQUEST_TYPES
)
APPLICATION_SCOPE["required_inputs"] = APP_CONTRACT["required_inputs"]
APPLICATION_SCOPE["routing_rules"] = APP_CONTRACT["routing_rules"]
RESPONSE_CONTRACT = APP_CONTRACT["response_contract"]
SUCCESS_CRITERIA = APP_CONTRACT["success_criteria"]
pretty(
    {
        "application_scope": APPLICATION_SCOPE,
        "response_contract": RESPONSE_CONTRACT,
        "success_criteria": SUCCESS_CRITERIA,
    }
)

## 7. Configure retrieval, tools, and response policy

The three imported contracts control the application. Edit one decision, rebuild the pipeline, and rerun the same request to compare its trace.

- `RETRIEVAL_CONTRACT` controls chunking, semantic/hybrid retrieval, reranking, and accepted evidence count.
- `CAPABILITY_CONTRACT` controls tool call order, required inputs, and failure handling.
- `APP_CONTRACT` controls supported requests, required response fields, and escalation routes.

The next cell displays settings derived from these contracts. Change the contracts themselves before rebuilding; changing only a display table does not change the runtime.

Check that your selected request type has the required tools and evidence sources.


In [ ]:
#@title Apply the validated Sprint 2 and Sprint 3 decisions
DEFAULT_FIELDCARE_PIPELINE_DESIGN = default_pipeline_design()
RERANK_CONFIG = DEFAULT_FIELDCARE_PIPELINE_DESIGN["rerank_config"]
RERANK_CONFIG.update(RETRIEVAL_CONTRACT["rerank_config"])
TOOL_PLAN_BY_REQUEST_TYPE = DEFAULT_FIELDCARE_PIPELINE_DESIGN[
    "tool_plan_by_request_type"
]
RETRIEVAL_ENABLED_BY_REQUEST_TYPE = DEFAULT_FIELDCARE_PIPELINE_DESIGN[
    "retrieval_required_by_request_type"
]
direct_names = {tool["name"] for tool in CAPABILITY_CONTRACT["direct_tools"]}
for rule in CAPABILITY_CONTRACT["orchestration_rules"]:
    TOOL_PLAN_BY_REQUEST_TYPE[rule["request_type"]] = [
        name for name in rule.get("call_order", []) if name in direct_names
    ]
    RETRIEVAL_ENABLED_BY_REQUEST_TYPE[rule["request_type"]] = (
        "search_service_docs" in rule.get("call_order", [])
    )
RESPONSE_POLICY = DEFAULT_FIELDCARE_PIPELINE_DESIGN["response_policy"]
show_rerank_config_guide(RERANK_CONFIG)
show_tool_plan(TOOL_PLAN_BY_REQUEST_TYPE)
show_retrieval_rules(RETRIEVAL_ENABLED_BY_REQUEST_TYPE)

## 8. Build the helper-backed FieldCare app

This section builds the application from the three validated sprint contracts. Five current-state and routing operations remain direct `ToolRegistry` tools. Each request executes its contracted call order. Document evidence comes from the FieldCare MCP server and is recorded in that request's trace.

Do not continue until both checks pass: the direct registry must expose exactly the five contracted tools, and MCP must advertise only `search_service_docs` and return document evidence.


In [ ]:
#@title Build the runnable FieldCare app
FIELDCARE_PIPELINE_DESIGN = assemble_pipeline_design(
    RERANK_CONFIG,
    TOOL_PLAN_BY_REQUEST_TYPE,
    RETRIEVAL_ENABLED_BY_REQUEST_TYPE,
    RESPONSE_POLICY,
    application_scope=APPLICATION_SCOPE,
    response_contract=RESPONSE_CONTRACT,
    success_criteria=SUCCESS_CRITERIA,
)
FIELDCARE_PIPELINE_DESIGN.update(
    {
        "app_contract": APP_CONTRACT,
        "retrieval_contract": RETRIEVAL_CONTRACT,
        "capability_contract": CAPABILITY_CONTRACT,
    }
)
fieldcare_pipeline = build_fieldcare_pipeline(fieldcare_env, FIELDCARE_PIPELINE_DESIGN)
pretty(fieldcare_pipeline.finalization_status)

In [ ]:
#@title Inspect the contracted boundaries
show_registered_tools(fieldcare_pipeline)
pretty({"request_plans": fieldcare_pipeline.contract_rules})

## 9. Run the guided end-to-end trace

This is the core project trace. One cell runs the full application path so you can see how the pieces fit together:

1. receive a technician request;
2. retrieve broad documentation candidates with `BM25Retriever`;
3. execute the planned registered tools through `ToolRegistry`;
4. rerank and filter accepted evidence;
5. assemble model-ready context;
6. classify response-control flags;
7. draft the application response object.

Expected output: the final response should be explainable from the displayed document IDs, tool statuses, and decision flags.

Check: before reading the final answer, inspect the accepted evidence and tool results. The answer is trustworthy only if those intermediate objects support it.


In [ ]:
#@title Run the guided FieldCare trace
GUIDED_REQUEST_ID = "REQ-FC-001"  #@param ["REQ-FC-001", "REQ-FC-002", "REQ-FC-003", "REQ-FC-006", "REQ-FC-007", "REQ-FC-010", "REQ-FC-011", "REQ-FC-012", "REQ-FC-013", "REQ-FC-014"]
guided_request = request_by_id(fieldcare_env, GUIDED_REQUEST_ID)

# Step 1: execute the contract plan, including MCP retrieval and direct tools.
guided_tool_results = execute_tool_plan(guided_request)
guided_baseline_candidates = copy.deepcopy(
    fieldcare_pipeline.last_execution["mcp_rows"]
)

# Step 2: rerank retrieved passages and filter them against equipment state.
guided_evidence = accepted_evidence(
    fieldcare_pipeline, guided_request, guided_tool_results
)

# Step 3: assemble the full passages, observed tool state, and application rules.
guided_context = assemble_fieldcare_context_from_parts(
    fieldcare_pipeline,
    guided_request,
    guided_tool_results,
    guided_evidence,
)

# Step 4: generate a real LLM response and validate its schema and citations.
guided_response = draft_application_response(guided_context)

show_orchestration_summary(
    guided_request,
    guided_baseline_candidates,
    guided_tool_results,
    guided_evidence,
    guided_context,
    guided_response,
)

## 10. Inspect the model call

The previous step completed RAG: retrieved passages and tool results were sent to a real LLM, which returned validated JSON. Inspect the model identifier, request ID, and token usage below. Citations must refer to chunks in this request's context. Schema validation checks structure; review the answer's claims against the evidence too.


In [ ]:
# The response above came from the live model call, not a template.
pretty(guided_response["model_call"])

## 11. Build your own functionality into the same path

Now reuse the same app for a request you define. Choose the closest request type, provide the known IDs if you have them, and run the exact same orchestration path. This is the safest way to extend the project: new behavior should still show documents, tools, context, flags, and response.

Action: choose one realistic user request. Keep it small enough that one trace can prove what changed.

Expected output: the same trace structure as the guided run, now driven by your custom request text and request type.

Check: your custom response should cite only evidence that appears in the accepted documents or tool results.


In [ ]:
#@title Choose the extension surface
show_extension_points()

In [ ]:
#@title Run a custom FieldCare request through the same app
CUSTOM_REQUEST_TEXT = "For EQ-FC-1001, the unit is overheating again after filter work. Can the technician close TCK-FC-9008 or should it be escalated?"  #@param {type:"string"}
CUSTOM_REQUEST_TYPE = "troubleshooting_plus_warranty"  #@param ["troubleshooting_plus_warranty", "retrieval_only_airflow", "warranty_only", "sensor_plus_warranty", "expired_warranty", "tool_unavailable", "conflicting_documentation", "reranking_distractor", "ticket_status_only", "repeat_fault_escalation", "safety_escalation", "ambiguous_troubleshooting", "incomplete_request", "unsupported_model", "unsupported_business_promise"]
CUSTOM_EQUIPMENT_ID = "EQ-FC-1001"  #@param {type:"string"}
CUSTOM_TICKET_ID = "TCK-FC-9008"  #@param {type:"string"}

custom_request = make_custom_request(
    request_text=CUSTOM_REQUEST_TEXT,
    request_type=CUSTOM_REQUEST_TYPE,
    equipment_id=CUSTOM_EQUIPMENT_ID,
    ticket_id=CUSTOM_TICKET_ID,
    expected_sources=[
        "retrieval",
        "equipment_tool",
        "maintenance_tool",
        "warranty_tool",
        "ticket_tool",
    ],
    preferred_behavior="Learner task: describe what the app should do before you trust the response.",
)

custom_baseline_candidates = []
custom_tool_results = execute_tool_plan(custom_request)
custom_baseline_candidates = copy.deepcopy(
    fieldcare_pipeline.last_execution["mcp_rows"]
)
custom_evidence = accepted_evidence(
    fieldcare_pipeline, custom_request, custom_tool_results
)
custom_context = assemble_fieldcare_context_from_parts(
    fieldcare_pipeline,
    custom_request,
    custom_tool_results,
    custom_evidence,
)
custom_response = draft_application_response(custom_context)

show_orchestration_summary(
    custom_request,
    custom_baseline_candidates,
    custom_tool_results,
    custom_evidence,
    custom_context,
    custom_response,
)

## 12. Stress-test one edge case

Edge cases are how you find the limits of your app. Pick one case, run it, and inspect the same trace before changing anything. The goal is not to make every case perfect immediately. The goal is to see where the path breaks and choose one targeted improvement.

Action: run one edge case without editing the code first.

Expected output: expected behavior, required evidence, the actual context trace, and the response object.

Check: identify the first missing or weak surface: retrieval, reranking, tool plan, response policy, or response assembly.


In [ ]:
#@title Run one edge-case trace
EDGE_CASE_EVAL_ID = "EVAL-FC-006"  #@param ["EVAL-FC-006", "EVAL-FC-007", "EVAL-FC-008", "EVAL-FC-009", "EVAL-FC-010", "EVAL-FC-011", "EVAL-FC-012", "EVAL-FC-013", "EVAL-FC-015", "EVAL-FC-016"]
edge_case_run = run_eval_case(fieldcare_pipeline, EDGE_CASE_EVAL_ID)

display(Markdown(f"### Edge case `{EDGE_CASE_EVAL_ID}`"))
pretty(
    {
        "expected_behavior": edge_case_run["eval_case"]["expected_behavior"],
        "required_doc_ids": edge_case_run["eval_case"]["required_doc_ids"],
        "required_tool_calls": edge_case_run["eval_case"]["required_tool_calls"],
        "expected_response_flags": edge_case_run["eval_case"][
            "expected_response_flags"
        ],
    }
)
show_context_trace(edge_case_run["context"])
pretty(edge_case_run["response"])

## 13. Use the evaluation map for debugging

The reusable cases are a debugging map, not the main project experience. Use them after you understand the end-to-end trace. Look for the first missing surface: document evidence, tool call, decision flag, or rejected legacy evidence.

Action: scan the failing rows before changing anything. Pick one failure pattern, not every failure.

Expected output: one row per reusable case with pass/fail evidence for documents, tools, flags, and rejected legacy docs.

Check: a useful next edit should improve a named row or pattern in this table.


In [ ]:
#@title Build a debugging map from all reusable cases
evaluation_results = evaluate_fieldcare_pipeline(fieldcare_pipeline)
evaluation_df = pd.DataFrame(evaluation_results)
show_evaluation_results(evaluation_df)

## 14. Plan one targeted improvement

Finish by choosing one change you can explain. A strong improvement has before-and-after evidence: different accepted `doc_id`s, different tool state, a clearer decision flag, or a safer response object.

Your project task answers should be easy to point to: the scoped app boundary, the response contract, the complete trace, the edge-case trace, and this improvement note.

Action: fill in every `Learner task:` field after you make and rerun one change.

Expected output: a short project-improvement record that you or a peer can understand.

Check: your after-evidence should name concrete `doc_id`s, tool statuses, or decision flags. Avoid vague claims like "it is better now."


In [ ]:
#@title Run and record a retrieval experiment
IMPROVEMENT_ACCEPTED_K = (
    None  # Set a different positive integer, then inspect both runs.
)
PROJECT_IMPROVEMENT_NOTE = {
    "before_evidence": "Learner task: run an experiment",
    "after_evidence": "Learner task: inspect the result",
}
if IMPROVEMENT_ACCEPTED_K is not None:
    before_config = copy.deepcopy(
        fieldcare_pipeline.retrieval_contract["selected_config"]
    )
    if (
        type(IMPROVEMENT_ACCEPTED_K) is not int
        or IMPROVEMENT_ACCEPTED_K < 1
        or IMPROVEMENT_ACCEPTED_K == before_config["accepted_k"]
    ):
        raise ValueError("Choose a different positive accepted_k.")
    before_run = run_fieldcare_app(GUIDED_REQUEST_ID)
    fieldcare_pipeline.retrieval_contract["selected_config"]["accepted_k"] = (
        IMPROVEMENT_ACCEPTED_K
    )
    after_run = run_fieldcare_app(GUIDED_REQUEST_ID)
    before_ids = [row["doc_id"] for row in before_run["context"]["retrieved_evidence"]]
    after_ids = [row["doc_id"] for row in after_run["context"]["retrieved_evidence"]]
    PROJECT_IMPROVEMENT_NOTE = {
        "request_or_edge_case": GUIDED_REQUEST_ID,
        "weak_behavior_observed": f"Baseline accepted {len(before_ids)} documents; inspect whether that evidence set is appropriate.",
        "first_surface_to_inspect": "retrieval",
        "one_change_to_try": f"Change accepted_k from {before_config['accepted_k']} to {IMPROVEMENT_ACCEPTED_K}.",
        "before_evidence": json.dumps(before_ids),
        "after_evidence": json.dumps(after_ids),
        "config_before": before_config,
        "config_after": copy.deepcopy(
            fieldcare_pipeline.retrieval_contract["selected_config"]
        ),
        "before_trace": before_run["context"]["execution_trace"],
        "after_trace": after_run["context"]["execution_trace"],
        "what_the_project_can_now_claim": f"This measured change accepted {len(after_ids)} documents instead of {len(before_ids)}; this alone does not prove answer quality improved.",
    }
    evaluation_results = evaluate_fieldcare_pipeline(fieldcare_pipeline)
    evaluation_df = pd.DataFrame(evaluation_results)
pretty(PROJECT_IMPROVEMENT_NOTE)

## 15. Answer the three project checks

Use this section after you have run the guided trace, one custom request, one edge case, and the evaluation map. The three answers and answer signals appear in one place so you can revise the notebook until the evidence is visible.

Action: run the cell and read each project check from top to bottom.

Expected output: three project task answers with answer signals and actual evidence.

Check: if evidence is missing, revise the earlier section named by that check and rerun this cell.


In [ ]:
#@title Answer the three project checks
improvement_fields_completed = {
    key: not str(value).startswith("Learner task:")
    for key, value in PROJECT_IMPROVEMENT_NOTE.items()
}

guided_expected_docs = {
    "DOC-FC-TS-001",
    "DOC-FC-MP-014",
    "DOC-FC-WAR-004",
    "DOC-FC-ESC-007",
}
guided_expected_tools = {
    "get_equipment_record",
    "get_maintenance_history",
    "get_warranty_status",
    "get_ticket_status",
}
guided_actual_docs = {row["doc_id"] for row in guided_context["retrieved_evidence"]}
guided_actual_tools = {row["tool_name"] for row in guided_context["tool_results"]}
guided_actual_flags = set(guided_response["decision_flags"])

edge_expected_docs = set(edge_case_run["eval_case"]["required_doc_ids"])
edge_expected_tools = set(edge_case_run["eval_case"]["required_tool_calls"])
edge_expected_flags = set(edge_case_run["eval_case"]["expected_response_flags"])
edge_forbidden_docs = set(edge_case_run["eval_case"]["must_not_use_doc_ids"])
edge_actual_docs = {
    row["doc_id"] for row in edge_case_run["context"]["retrieved_evidence"]
}
edge_actual_tools = {
    row["tool_name"] for row in edge_case_run["context"]["tool_results"]
}
edge_actual_flags = set(edge_case_run["response"]["decision_flags"])

PROJECT_TASK_ANSWERS = {
    "task_1_integrated_request": {
        "actual_request_id": GUIDED_REQUEST_ID,
        "accepted_doc_ids": sorted(guided_actual_docs),
        "tool_names": sorted(guided_actual_tools),
        "decision_flags": sorted(guided_actual_flags),
        "evidence_seen": {
            "used_req_fc_001": GUIDED_REQUEST_ID == "REQ-FC-001",
            "has_required_docs": guided_expected_docs.issubset(guided_actual_docs),
            "has_required_tools": guided_expected_tools.issubset(guided_actual_tools),
            "has_escalation_or_review_signal": bool(
                guided_actual_flags
                & {"escalate", "safety_escalation", "history_changes_answer"}
            ),
        },
    },
    "task_2_edge_case": {
        "eval_id": EDGE_CASE_EVAL_ID,
        "expected_behavior": edge_case_run["eval_case"]["expected_behavior"],
        "accepted_doc_ids": sorted(edge_actual_docs),
        "tool_names": sorted(edge_actual_tools),
        "decision_flags": sorted(edge_actual_flags),
        "evidence_seen": {
            "has_required_docs": edge_expected_docs.issubset(edge_actual_docs),
            "avoids_forbidden_docs": not bool(edge_forbidden_docs & edge_actual_docs),
            "has_required_tools": edge_expected_tools.issubset(edge_actual_tools),
            "has_expected_flags": edge_expected_flags.issubset(edge_actual_flags),
        },
    },
    "task_3_improvement": {
        "improvement_note": PROJECT_IMPROVEMENT_NOTE,
        "fields_completed": improvement_fields_completed,
        "evaluation_passes": f"{int(evaluation_df['pipeline_pass'].sum())}/{len(evaluation_df)}",
        "evidence_seen": {
            "improvement_note_complete": all(improvement_fields_completed.values()),
            "has_before_after_evidence": all(
                improvement_fields_completed[field]
                for field in [
                    "before_evidence",
                    "after_evidence",
                    "what_the_project_can_now_claim",
                ]
            ),
        },
    },
}

PROJECT_RESPONSES = {
    "task_1_integrated_request": "",
    "task_2_edge_case": "",
    "task_3_improvement": "",
}
pretty({"your_task_answers": PROJECT_TASK_ANSWERS})


def reveal_project_answer_signals():
    if not all(
        len(value.split()) >= 30 for value in PROJECT_RESPONSES.values()
    ) or not all(improvement_fields_completed.values()):
        print(
            "Complete the three written responses and the measured experiment before comparison."
        )
        return
    PROJECT_ANSWER_SIGNALS = {
        "task_1_integrated_request": {
            "question": "Can FieldCare handle the integrated overheating, warranty, and escalation request?",
            "answer": "`REQ-FC-001` should combine current HX troubleshooting, filter, warranty, and escalation documents with equipment, maintenance, warranty, and ticket tools. Repeat-fault evidence should lead to escalation or supervisor review rather than routine closure.",
            "look_for": {
                "request_id": "REQ-FC-001",
                "required_doc_ids": sorted(guided_expected_docs),
                "required_tools": sorted(guided_expected_tools),
                "useful_flags": [
                    "cite_evidence",
                    "mention_coverage_condition",
                    "escalate",
                ],
            },
        },
        "task_2_edge_case": {
            "question": "Can FieldCare handle the selected edge case honestly?",
            "answer": "The trace should match the selected eval case: required documents are accepted, forbidden documents do not drive the answer, required tools run, and expected decision flags appear. Missing signals tell you which surface to inspect next.",
            "look_for": {
                "eval_id": EDGE_CASE_EVAL_ID,
                "required_doc_ids": sorted(edge_expected_docs),
                "required_tools": sorted(edge_expected_tools),
                "forbidden_doc_ids": sorted(edge_forbidden_docs),
                "expected_flags": sorted(edge_expected_flags),
            },
        },
        "task_3_improvement": {
            "question": "What changed after your targeted improvement?",
            "answer": "The note names one changed surface and shows before-and-after evidence from concrete `doc_id`s, tool statuses, decision flags, or response fields. It also states the strongest honest next-version claim.",
            "look_for": {
                "all_note_fields_completed": all(improvement_fields_completed.values()),
                "fields": list(PROJECT_IMPROVEMENT_NOTE),
            },
        },
    }

    pretty({"answer_signals": PROJECT_ANSWER_SIGNALS})

## 16. Export the final submission manifest

The manifest ties the three sprint artifacts to this final run. It records immutable file digests, the integrated trace, the evaluation result, one before-and-after improvement, and honest limitations. Complete the improvement note first; unresolved placeholders intentionally prevent export.


In [ ]:
from fieldcare import artifact_record, write_fieldcare_artifact

if not all(improvement_fields_completed.values()):
    print(
        "Complete every PROJECT_IMPROVEMENT_NOTE field, rerun the improvement, then rerun this cell."
    )
else:
    FIELDCARE_SUBMISSION_MANIFEST = {
        "schema_version": "1.0",
        "project_id": "fieldcare",
        "artifacts": [artifact_record(name) for name in ARTIFACT_TYPES],
        "final_trace": {
            "request_id": GUIDED_REQUEST_ID,
            "model_call": guided_response["model_call"],
            "accepted_doc_ids": sorted(guided_actual_docs),
            "direct_tools": sorted(guided_actual_tools),
            "mcp_tool": next(
                (
                    call["tool"]
                    for call in guided_context["execution_trace"]
                    if call["boundary"] == "mcp"
                ),
                None,
            ),
            "calls": guided_context["execution_trace"],
            "decision_flags": sorted(guided_actual_flags),
        },
        "evaluation_summary": {
            "passed": int(evaluation_df["pipeline_pass"].sum()),
            "total": len(evaluation_df),
            "contract_passed": int(evaluation_df["contract_pass"].sum()),
            "catalog_cases_in_scope": int(
                evaluation_df["catalog_case_applicable"].sum()
            ),
            "selected_edge_case": EDGE_CASE_EVAL_ID,
        },
        "selected_improvement": PROJECT_IMPROVEMENT_NOTE,
        "limitations": [
            "Model generation is live; retrieved documents and tool records remain the sources of truth.",
            "Evaluation covers the supplied course dataset only; it is not proof of production reliability.",
            "passed counts the original catalog expectations; contract_passed also credits correct refusals outside the declared scope.",
        ],
    }
    SUBMISSION_MANIFEST_PATH = write_fieldcare_artifact(
        FIELDCARE_SUBMISSION_MANIFEST,
        "submission_manifest",
        "fieldcare_submission_manifest.json",
    )
    print(f"Validated final manifest: {SUBMISSION_MANIFEST_PATH}")
    try:
        from google.colab import files

        files.download(str(SUBMISSION_MANIFEST_PATH))
    except ImportError:
        pass

## Watch out for

- Do not edit the hidden setup cell unless you are repairing the notebook itself. Student project work should happen in the visible configuration, request, testing, and evidence cells.
- Do not trust a final response before checking the accepted `doc_id`s and tool statuses that produced it.
- Do not use warranty evidence alone for troubleshooting advice. Good FieldCare answers usually combine service documents with equipment, maintenance, warranty, and ticket state.
- Do not tune several surfaces at once. Change one retrieval, tool-plan, reranking, policy, or response-assembly decision, then rerun the same request.

## Quick reference

| Surface | Use it when | Evidence to check |
|---|---|---|
| `APP_CONTRACT` | Scope, required fields, or routes need changing. | Scope checks, response keys, and escalation path. |
| `RETRIEVAL_CONTRACT` | Retrieved evidence needs improving. | Backend, candidate scores, and accepted chunk IDs. |
| `CAPABILITY_CONTRACT` | The app needs different tool ordering or failure handling. | Call order, statuses, and recovery actions. |
| `PROJECT_IMPROVEMENT_NOTE` | You need to explain what changed and why it matters. | Before-and-after docs, tools, flags, or response fields. |

## Summary

You built and inspected a helper-backed RAG plus tool-use application for FieldCare. The important project habit is not just getting an answer: it is making the answer traceable through request scope, retrieved documentation, structured tool state, response-control flags, edge-case testing, and a clear next improvement.
